# Binary Target Mapping

This notebook loads the verified full prepared cohort and constructs the Any Readmission target. The intended prediction point is near discharge.

## 1. Setup

In [ ]:
import sys
from collections import Counter
from pathlib import Path

from IPython.display import Markdown, display

working_directory = Path.cwd().resolve()
project_root = next(
    directory
    for directory in (working_directory, *working_directory.parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preparation.data_loading import TARGET_COLUMN, TARGET_LABELS
from src.preparation.dataset_export import (
    CLASS_COLUMN,
    CLASS_LABELS,
    build_model_dataset,
    load_full_prepared,
    load_target_mapping,
    map_binary_targets,
)

## 2. Load Full Prepared Data

The loader checks the saved CSV against its preparation manifest and the configured predictor order. The cohort table retains both identifiers and the three original target values.

In [ ]:
prepared_columns, prepared_rows = load_full_prepared()
source_counts = Counter(row[TARGET_COLUMN] for row in prepared_rows)

print(f"Prepared cohort: {len(prepared_rows):,} encounters, {len(prepared_columns)} columns")
print("Original labels:", {label: source_counts[label] for label in TARGET_LABELS})

## 3. Binary Target Mapping

Both recorded readmission categories map to the positive class. The negative class indicates that no readmission was recorded. Counts below use the full eligible cohort as the denominator and describe class prevalence before partitioning.

In [ ]:
target_mapping = load_target_mapping()
mapped_rows = map_binary_targets(prepared_rows)
model_columns, model_rows = build_model_dataset(prepared_columns, mapped_rows)
class_counts = Counter(row[CLASS_COLUMN] for row in model_rows)

table = [
    "| Original label | Model class | Encounters |",
    "| --- | --- | ---: |",
    *(f"| `{label}` | `{target_mapping[label]}` | {source_counts[label]:,} |" for label in TARGET_LABELS),
    "",
    "| Model class | Encounters | Share of cohort |",
    "| --- | ---: | ---: |",
    *(
        f"| `{label}` | {class_counts[label]:,} | {class_counts[label] / len(model_rows):.2%} |"
        for label in CLASS_LABELS
    ),
]
display(Markdown("\n".join(table)))
print(f"Internal columns: {len(prepared_columns) + 1}")
print(f"Model columns: {len(model_columns)}, with {model_columns[-1]!r} last")

**Observation.** Mapping preserves the cohort size. The internal records retain encounter and patient identifiers and the original labels for partitioning.

**Implication.** The model view contains the 32 ordered predictors followed by `class`.

**Limitation.** The positive class represents readmission recorded within the source data's observation coverage. The binary target combines the two source timing categories.